# ⚡ Task 1: Electric Vehicle Data Import, Exploration & Statistical Analysis
### Revised Bloom's Taxonomy Level: K2 (Understand), K3 (Apply)
**Course:** Python for Data Science & Machine Learning  
**Term Work / Self Learning:** 4 Hours (3 Hours Lab Work + 1 Hour Group Discussion & Demonstration)  
**Submission Date:** 31.08.2026 (Friday)  

---
## Notebook Outline:
1. **Self-Learning Reference:** NumPy, Pandas, Matplotlib, Seaborn
2. **Data Import & Initial Exploration** (`head`, `info`, `describe`, missing values, duplicates)
3. **Data Preprocessing & Cleaning** (deduplication, median imputation, IQR outlier detection)
4. **Feature Engineering** (Range efficiency, cost per range mile, Z-score standardization)
5. **Statistical Measures & Relational Matrices** (Mean, Median, Mode, Variance, Covariance, Pearson Correlation)
6. **Probability Distributions & Normality Testing** (Skewness, Kurtosis, Shapiro-Wilk Test)
7. **Publication-Grade Data Visualizations** (Histograms with KDE, Boxplots, Scatter, Heatmap, Q-Q Plot, Bar charts)
8. **Demonstration Protocol & Viva-Voce Q&A**

### 1. Library Imports and Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Styling setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 150
print('Libraries successfully loaded.')

### 2. Data Ingestion & Structural Exploration

In [ ]:
DATA_PATH = '../data/ev_dataset.csv' if os.path.exists('../data/ev_dataset.csv') else 'ev_dataset.csv'
df_raw = pd.read_csv(DATA_PATH)
print(f'Dataset Dimensions: {df_raw.shape[0]} Rows, {df_raw.shape[1]} Columns')
df_raw.head(5)

In [ ]:
# Schema and missing values audit
print('--- Dataset Info ---')
df_raw.info()
print('\n--- Missing Values Audit ---')
null_summary = pd.DataFrame({
    'Missing_Count': df_raw.isnull().sum(),
    'Percentage (%)': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
null_summary[null_summary['Missing_Count'] > 0]

### 3. Data Preprocessing & Cleaning Pipeline

In [ ]:
df = df_raw.copy()

# 1. Deduplication
init_len = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Dropped {init_len - len(df)} duplicates. Remaining rows: {len(df)}')

# 2. Median Imputation for Skewed Features
impute_cols = ['Battery_Capacity_kWh', 'Base_MSRP_USD', 'Charging_Time_Hours']
for col in impute_cols:
    med = df[col].median()
    df[col] = df[col].fillna(med)
    print(f'Imputed {col} with median: {med:.2f}')

# 3. Outlier Audit using IQR Method
print('\n--- Outlier Detection via IQR Method ---')
for col in ['Battery_Capacity_kWh', 'Electric_Range_Miles', 'Base_MSRP_USD']:
    q25 = df[col].quantile(0.25)
    q75 = df[col].quantile(0.75)
    iqr = q75 - q25
    lower = q25 - 1.5 * iqr
    upper = q75 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f'{col}: Bounds [{lower:.2f}, {upper:.2f}] | Outliers: {len(outliers)}')

### 4. Feature Engineering

In [ ]:
# Engineering Range Efficiency and Cost per Mile of Range
df['Range_Efficiency_Miles_per_kWh'] = (df['Electric_Range_Miles'] / df['Battery_Capacity_kWh']).round(2)
df['Cost_per_Range_Mile_USD'] = (df['Base_MSRP_USD'] / df['Electric_Range_Miles']).round(2)
df[['Make', 'Model', 'Range_Efficiency_Miles_per_kWh', 'Cost_per_Range_Mile_USD']].head()

### 5. Statistical Measures & Relational Matrices

In [ ]:
num_cols = [
    'Battery_Capacity_kWh', 'Electric_Range_Miles', 'Base_MSRP_USD',
    'Charging_Time_Hours', 'Energy_Consumption_Wh_km', 'Acceleration_0_100_s',
    'Top_Speed_kmh', 'Range_Efficiency_Miles_per_kWh'
]

stat_rows = []
for col in num_cols:
    s = df[col]
    mode_val = s.mode().iloc[0] if not s.mode().empty else np.nan
    shapiro_stat, p_val = stats.shapiro(s[:500])
    stat_rows.append({
        'Feature': col,
        'Mean': round(s.mean(), 2),
        'Median': round(s.median(), 2),
        'Mode': round(mode_val, 2),
        'Variance': round(s.var(ddof=1), 2),
        'Std_Dev': round(s.std(ddof=1), 2),
        'Range': round(s.max() - s.min(), 2),
        'IQR': round(s.quantile(0.75) - s.quantile(0.25), 2),
        'Skewness': round(stats.skew(s), 3),
        'Kurtosis': round(stats.kurtosis(s), 3),
        'Normality': 'Normal (p>0.05)' if p_val > 0.05 else 'Non-Normal (p<=0.05)'
    })

summary_df = pd.DataFrame(stat_rows)
summary_df

In [ ]:
# Pearson Correlation Matrix
corr_matrix = df[num_cols].corr(method='pearson')
corr_matrix.round(3)

### 6. Visual Analytics & Exploratory Insights

In [ ]:
# Figure 1: Univariate Histograms & Boxplots
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.histplot(df['Electric_Range_Miles'], kde=True, ax=axes[0, 0], color='#1e3a8a')
axes[0, 0].set_title('Electric Range Distribution')
sns.histplot(df['Battery_Capacity_kWh'], kde=True, ax=axes[0, 1], color='#0d9488')
axes[0, 1].set_title('Battery Capacity Distribution')
sns.boxplot(data=df, x='EV_Type', y='Electric_Range_Miles', ax=axes[1, 0], palette=['#3b82f6', '#10b981'])
axes[1, 0].set_title('Range by Powertrain Type')
sns.boxplot(data=df, x='EV_Type', y='Base_MSRP_USD', ax=axes[1, 1], palette=['#3b82f6', '#10b981'])
axes[1, 1].set_title('MSRP ($) by Powertrain Type')
plt.tight_layout()
plt.show()

In [ ]:
# Figure 2: Scatter Plot of Range vs Battery Capacity
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Battery_Capacity_kWh', y='Electric_Range_Miles', hue='EV_Type', size='Base_MSRP_USD', sizes=(40, 200), palette={'BEV': '#2563eb', 'PHEV': '#059669'})
sns.regplot(data=df, x='Battery_Capacity_kWh', y='Electric_Range_Miles', scatter=False, color='#dc2626', line_kws={'linestyle': '--', 'label': 'Global Fit'})
plt.title('Electric Range vs. Battery Capacity (r = 0.899)', fontweight='bold')
plt.legend()
plt.show()

In [ ]:
# Figure 3: Correlation Heatmap
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Pearson Correlation Matrix Heatmap', fontweight='bold')
plt.show()

### 7. Group Discussion & Demonstration Guide
- **Theme 1:** Battery Capacity vs Curb Weight and Energy Efficiency.
- **Theme 2:** Bimodal Powertrain Segmentation (PHEV urban commuting vs BEV touring).
- **Theme 3:** Ethical Handling of Financial Outliers in ML Training Sets.